# Agentic RAG: turn a fixed pipeline into a decision-making loop

Static RAG is a **fixed** pipeline — embed the query once, retrieve top-k once, stuff the
passages into a prompt, generate once. That single shot cannot do **multi-step** retrieval,
pick **which** source, run a **calculation** on what it retrieved, or notice a miss and try
again. A **compound** query needs all of that.

This notebook builds a real **ReAct** control loop (Thought → Action → Observation, repeat),
gives it **real tools** (a dense retriever, a calculator, a router), and shows it solve a
compound query that static single-shot RAG structurally **cannot** — then measures the cost.

> **Honesty (say it up front).** The **control loop**, the **tools**, and the **routing** are
> real and measured. The one illustrative piece is the **action-selection policy** — in
> production an LLM decides the next action from the running trace; this env is encoder-only,
> so we use a deterministic rule-based stand-in that makes the *same* decisions an LLM would.
> The tools it invokes (retrieval, arithmetic, routing) run for real. This is flagged again at
> every step.

> **Step 1 — import the canonical, verified code.** Everything below comes from
> `agentic_rag.py` next to this notebook — the *same* module the concept page and the figures
> use, which in turn reuses chapter 5's real `DenseRetriever` (all-MiniLM). Nothing is
> re-implemented in the notebook; we drive the real objects and print real numbers. The banner
> line reports the encoder backend and the accelerator (the encoder itself is CPU-pinned).

In [1]:
from agentic_rag import (
    COMPOUND_QUERY,
    Router,
    build_agent,
    compound_orbit_policy,
    never_finishing_policy,
    full_corpus,
    safe_eval,
    static_rag,
)

corpus = full_corpus()
agent, tools, retriever_tool = build_agent(corpus)
dense = retriever_tool._dense  # the shared all-MiniLM encoder, reused by the router
router = Router(tools, dense)

print('corpus passages :', len(corpus))
print('dense lens      :', retriever_tool.backend)
print('tools           :', [t.name for t in tools])
print('step budget     :', agent.max_steps)

corpus passages : 11
dense lens      : sentence-transformers/all-MiniLM-L6-v2
tools           : ['retrieve', 'calculator']
step budget     : 6


> **Step 2 — meet the corpus and the compound query.** The corpus is the shared Helios-7
> knowledge base (chapter 1 + chapter 5's blind-spot passages). The query we will use asks
> **two things at once**, and the first part needs *arithmetic on a retrieved number* — which
> is exactly what a fixed pipeline has no place to do.

In [2]:
for i, passage in enumerate(corpus):
    print(f'  doc[{i:>2}] {passage}')
print()
print('COMPOUND QUERY:', COMPOUND_QUERY)

  doc[ 0] The Helios-7 satellite was launched on March 3rd, 2024 from the Kourou spaceport.
  doc[ 1] Helios-7 carries a hyperspectral imager with a ground resolution of 4 meters.
  doc[ 2] The project lead for Helios-7 is Dr. Amara Okoye, based in the Nairobi office.
  doc[ 3] Helios-7 completes one orbit of Earth every 97 minutes in a sun-synchronous orbit.
  doc[ 4] Photosynthesis converts carbon dioxide and water into glucose using sunlight.
  doc[ 5] The Eiffel Tower in Paris was completed in 1889 for the World's Fair.
  doc[ 6] A standard chessboard has 64 squares arranged in an 8 by 8 grid.
  doc[ 7] Water boils at 100 degrees Celsius at standard atmospheric pressure.
  doc[ 8] Error E-4011 appeared in the Helios-7 telemetry stream.
  doc[ 9] Climbing steadily, Helios-7 rose skyward moments past liftoff.
  doc[10] The Helios-7 ground team spent the afternoon investigating several telemetry errors and console warnings.

COMPOUND QUERY: How many complete orbits does Helios-7 make 

## 1) Where static single-shot RAG breaks

Static RAG embeds the query **once**, retrieves top-k **once**, and answers from those
passages. It can surface the passage that *contains* the orbit period and the one with the
resolution — but it has **no step to divide** 1440 minutes by the period, so the *count* part
of the compound query is unanswerable in the pipeline.

> **Step 3 — run static RAG and watch the count come back empty.** We honestly model the
> pipeline's limit: it reads the resolution off the retrieved passages, but `orbit_count` is
> `None` because there is no arithmetic step. The assert makes the limitation a checked fact,
> not a claim.

In [3]:
static = static_rag(COMPOUND_QUERY, dense, corpus, k=3)
print('retrieved (top-3):')
for p in static.retrieved:
    print('  -', p)
print()
print('resolution it can read off :', static.resolution)
print('orbit count it can compute  :', static.orbit_count, ' (no arithmetic step in the pipeline)')
assert static.orbit_count is None, 'static single-shot RAG has no step to compute the orbit count'
print('\n-> static RAG surfaces facts but cannot DIVIDE 1440 by the period; the count is unanswerable.')

retrieved (top-3):
  - Helios-7 carries a hyperspectral imager with a ground resolution of 4 meters.
  - Helios-7 completes one orbit of Earth every 97 minutes in a sun-synchronous orbit.
  - The Helios-7 satellite was launched on March 3rd, 2024 from the Kourou spaceport.

resolution it can read off : 4 meters
orbit count it can compute  : None  (no arithmetic step in the pipeline)

-> static RAG surfaces facts but cannot DIVIDE 1440 by the period; the count is unanswerable.


## 2) The tools are real — poke them directly

Before we hand the tools to the agent, run each one by hand so you trust it. The retriever is
chapter 5's real dense index; the calculator is a **safe** AST evaluator (never `eval`).

> **Step 4 — the retriever tool: a focused sub-question, a real passage back.** The agent
> never sends the whole compound query to the retriever — it sends one **sharp sub-question**
> at a time. That is why it succeeds where one-shot retrieval fails. Here we ask for the orbit
> period and get the right passage, with its real cosine score.

In [4]:
obs = retriever_tool('Helios-7 orbit period in minutes')
idx, cos = obs.value
print('retriever returned doc[%d] (cos=%.3f):' % (idx, cos))
print('  ', obs.text)
assert '97 minutes' in obs.text, 'the orbit-period sub-question must retrieve the 97-minute passage'

retriever returned doc[3] (cos=0.799):
   Helios-7 completes one orbit of Earth every 97 minutes in a sun-synchronous orbit.


> **Step 5 — the calculator tool: the step static RAG lacks (and why it is safe).** The
> calculator parses its input to an AST and evaluates **only** numeric literals and basic
> operators — a name, an attribute, or a function call raises. That is the standard safe-eval
> pattern: a tool can take agent-provided input without the code-execution risk of bare
> `eval()`. We compute the orbits/day, and confirm the disallowed case is rejected.

In [5]:
print('1440 / 97 =', safe_eval('1440 / 97'))
print('1440 // 97 =', safe_eval('1440 // 97'), ' (complete orbits per day)')
assert safe_eval('1440 // 97') == 14, 'Helios-7 makes 14 COMPLETE orbits per day'

# the safety property: anything that is not pure arithmetic is refused
try:
    safe_eval('__import__("os").system("echo hi")')
    raise SystemExit('SECURITY BUG: safe_eval executed code it should have refused')
except ValueError as e:
    print('refused non-arithmetic input:', str(e)[:60], '...')

1440 / 97 = 14.845360824742269
1440 // 97 = 14.0  (complete orbits per day)
refused non-arithmetic input: disallowed expression element: Call(func=Attribute(value=Cal ...


## 3) The ReAct loop — watch it iterate

Now hand the tools to the agent. The loop is the lesson: at each step the policy returns a
**Thought** (reasoning), an **Action** (a tool + input), the agent **runs the tool for real**,
reads the **Observation**, and repeats — until it emits `finish` or hits the step budget.

> **Step 6 — run the agent and print the full trace.** This runs the *real* control loop over
> the *real* tools. Read the trace top to bottom: retrieve the period → compute 1440/97 →
> retrieve the resolution → finish. Each `Observation` is a tool's actual output; only the
> `Thought` text is the illustrative stand-in for an LLM policy.

In [6]:
result = agent.run(COMPOUND_QUERY, compound_orbit_policy)
for i, step in enumerate(result.steps, start=1):
    print(f'step {i}')
    print(f'  Thought:     {step.thought}')
    print(f'  Action:      {step.action}({step.action_input!r})')
    print(f'  Observation: {step.observation}')
print()
print('FINAL ANSWER:', result.answer)
print('steps:', result.n_steps, '| tools:', result.tools_used, '| hit budget:', result.hit_budget)

step 1
  Thought:     The question has two parts. First I need the orbit period, then I can compute orbits per day.
  Action:      retrieve('Helios-7 orbit period in minutes')
  Observation: Helios-7 completes one orbit of Earth every 97 minutes in a sun-synchronous orbit.
step 2
  Thought:     The passage says the period is 97 minutes. A day has 1440 minutes, so I divide to get orbits per day.
  Action:      calculator('1440 / 97')
  Observation: 1440 / 97 = 14.8454
step 3
  Thought:     That gives the orbits per day; the whole-number part is the count of COMPLETE orbits. Now I still need the imager's ground resolution.
  Action:      retrieve('Helios-7 imager ground resolution')
  Observation: Helios-7 carries a hyperspectral imager with a ground resolution of 4 meters.
step 4
  Thought:     I now have both facts: the orbit count and the resolution. I can answer.
  Action:      finish('Helios-7 completes 14 full orbits per day (14.85 raw), and its imager has a ground resolution of 4 

> **Step 7 — assert the solve before believing it.** Correctness *before* the claim: the agent
> finished cleanly (not by exhausting the budget), used the right tool sequence, and assembled
> the right facts — **14** complete orbits (1440 // 97) and the **4-meter** resolution. If any
> assert fired, the trace above would be wrong.

In [7]:
assert not result.hit_budget, 'the agent should finish via `finish`, not the budget'
assert result.tools_used == ['retrieve', 'calculator', 'retrieve', 'finish'], result.tools_used
assert '14 full orbits' in result.answer, 'must compute 14 complete orbits'
assert '4 meters' in result.answer, 'must retrieve the 4-meter resolution'
print('agent solved a query static RAG could not: multi-step retrieve + compute + retrieve. All asserts passed.')

agent solved a query static RAG could not: multi-step retrieve + compute + retrieve. All asserts passed.


## 4) Routing — pick the right tool by scoring the query

"The agent picks the right source" is concretely a **scoring decision**: embed the query and
each tool's **description**, and route to the argmax. A fact-lookup lands on the retriever; a
math query lands on the calculator. This is a real cosine, using the same encoder.

> **Step 8 — route four probes and read the scores.** Two are clear (a person lookup, a launch
> date → retriever; an explicit multiplication → calculator). One — *"1440 divided by 97?"* —
> is nearly a **tie**: routing is a genuine decision that can be marginal, which is why real
> systems sometimes add a small margin or a tie-break. Every route is asserted.

In [8]:
probes = [
    ('Who is the project lead for Helios-7?', 'retrieve'),
    ('What is 1440 divided by 97?', 'calculator'),
    ('When was Helios-7 launched?', 'retrieve'),
    ('Compute 200 times 4.', 'calculator'),
]
print(f"{'query':<44} | {'route':>11} | retrieve / calculator")
print('-' * 84)
for q, expected in probes:
    scores = router.route_scores(q)
    chosen, _ = router.route(q)
    print(f'{q:<44} | {chosen.name:>11} | {scores[0]:+.3f} / {scores[1]:+.3f}')
    assert chosen.name == expected, f'mis-routed {q!r} to {chosen.name}'
print('\nall four probes routed as expected (fact -> retriever, math -> calculator).')

query                                        |       route | retrieve / calculator
------------------------------------------------------------------------------------
Who is the project lead for Helios-7?        |    retrieve | +0.629 / -0.017
What is 1440 divided by 97?                  |  calculator | +0.232 / +0.252
When was Helios-7 launched?                  |    retrieve | +0.730 / -0.015
Compute 200 times 4.                         |  calculator | +0.084 / +0.388

all four probes routed as expected (fact -> retriever, math -> calculator).


## 5) The pitfalls that bite (and their fixes)

The agent's power comes with real failure modes. The most dangerous is a loop that never
stops.

> **Step 9 — no step budget → infinite loop; the cap is the fix.** We run a deliberately
> **broken** policy that never emits `finish`. Without a budget this would hang forever; the
> `MAX_STEPS` cap turns the hang into a bounded stop. The assert proves the budget fired at
> exactly the cap.

In [9]:
broken = agent.run(COMPOUND_QUERY, never_finishing_policy)
print('broken policy ran', broken.n_steps, 'steps; hit budget:', broken.hit_budget)
print('final answer:', broken.answer)
assert broken.hit_budget, 'the never-finishing policy must be stopped by the budget'
assert broken.n_steps == agent.max_steps, 'it should run exactly MAX_STEPS, then stop'
print(f'\n-> the cap ({agent.max_steps}) turns an infinite loop into a bounded, debuggable stop.')

broken policy ran 6 steps; hit budget: True
final answer: (step budget exhausted -- no final answer)

-> the cap (6) turns an infinite loop into a bounded, debuggable stop.


> **Step 10 — the cost of agency: steps ≈ LLM calls.** The agent's power is not free. Static
> RAG is a single shot; the agent took several steps, and in production **each step is an LLM
> call**. That is the trade: reach for an agent when the query *needs* multi-step reasoning,
> and stay with static RAG when one shot suffices (the **over-agentic** pitfall — using an
> agent where a fixed pipeline would do).

In [10]:
static_steps = 1  # one retrieval + one generation = a single shot
agent_steps = result.n_steps
print(f'static single-shot RAG : {static_steps} pass')
print(f'ReAct agent            : {agent_steps} steps '
      f'({result.tools_used.count("retrieve")} retrievals, '
      f'{result.tools_used.count("calculator")} calc, 1 finish)')
print(f'ratio                  : {agent_steps}x the per-query work')
assert agent_steps > static_steps, 'the agent trades more per-query work for solving harder queries'

static single-shot RAG : 1 pass
ReAct agent            : 4 steps (2 retrievals, 1 calc, 1 finish)
ratio                  : 4x the per-query work


## Try it yourself

Before you run the next cell, **predict**. The compound query needs the orbit period, then a
division, then the resolution — the policy uses exactly **4 steps** (retrieve, calc, retrieve,
finish). Now suppose we give the agent a query that needs **only one fact** (no arithmetic),
say *"When was Helios-7 launched?"*, with a matching one-step policy.

**Predict:** how many steps will *that* run — more, fewer, or the same as the compound query's
4? And which tools will appear in `tools_used`?

Then run it and check. *(Hint: a single-fact question needs one retrieval and a finish — no
calculator, no second retrieval. The agent adapts its step count to the query; that adaptivity
is the whole point, and also why simple queries shouldn't pay the multi-step tax.)*

In [11]:
# a one-step policy for a single-fact query: retrieve once, then finish with the passage.
def single_fact_policy(query, steps):
    if len(steps) == 0:
        return ('I need one fact: the launch date.', 'retrieve', 'When was Helios-7 launched?')
    return ('I have the passage; I can answer.', 'finish', steps[-1].observation)

simple = agent.run('When was Helios-7 launched?', single_fact_policy)
print('simple query steps:', simple.n_steps, '| tools:', simple.tools_used)
print('answer:', simple.answer)
assert simple.n_steps == 2, 'a single-fact query should take just retrieve + finish'
assert simple.tools_used == ['retrieve', 'finish'], simple.tools_used
assert 'March 3rd, 2024' in simple.answer, 'the launch-date passage should be retrieved'
print('\n-> 2 steps, not 4: the agent adapts its work to the query. Fewer facts, fewer steps.')

simple query steps: 2 | tools: ['retrieve', 'finish']
answer: The Helios-7 satellite was launched on March 3rd, 2024 from the Kourou spaceport.

-> 2 steps, not 4: the agent adapts its work to the query. Fewer facts, fewer steps.


## What we saw

- **Static single-shot RAG could not answer the compound query** — it surfaced facts but had
  no step to compute `1440 / 97`, so `orbit_count` came back `None`.
- **The ReAct agent solved it in 4 real steps** — retrieve the period → compute 14 orbits →
  retrieve the resolution → finish, asserting the right facts before claiming them.
- **Routing is a scoring decision** — the query's cosine to each tool's description picks the
  tool; sometimes (the `1440 / 97` probe) it is a near-tie.
- **A step budget is not optional** — a policy that never finishes loops forever without it; the
  cap turns a hang into a bounded stop.
- **Agency costs steps** — the agent did 4× the per-query work of static RAG, and adapts that
  count to the query (2 steps for a single-fact question). Use an agent when the query *needs*
  multi-step reasoning; stay static when one shot suffices.

**What was real vs illustrative:** the control loop, the tools (dense retriever + safe
calculator), the routing, the retrieval, and the arithmetic are all **real and measured**. The
only illustrative piece is the **action-selection policy** (an LLM's job in production) and the
`Thought` text — a deterministic stand-in that makes the same decisions an LLM would, so the
whole loop is reproducible with no generative model.